# Oocyte Annotation Analysis

This notebook analyses the oocyte annotations stored as GeoJSON files in
`data/dataset_28_04/` (updated 28 April 2026). Each file corresponds to a
group of coral slices (H&E stained) from a single specimen, and each feature
inside the file represents a single oocyte annotation labelled with its
oosorption stage (Stage 0 – Stage 4).

The notebook answers questions such as:

1.  How many oocytes were annotated per slice file?
2.  What is the overall distribution of oosorption stages?
3.  How does stage composition vary across samples?
4.  What are the geometric characteristics (area, perimeter) of oocytes at
    each stage?

The annotations were exported from QuPath.  Feature geometries are either
`Polygon` / `MultiPolygon` (filled regions) or `LineString` (traced contours
that form closed shapes); both are converted to `shapely` polygons before
any geometric computation is performed.


In [ ]:
%load_ext autoreload
%autoreload 2

import json
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import shape, Polygon, LineString, MultiPolygon

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_rows", 200)


In [ ]:
# Path to the labelled GeoJSON folder.  Update this if you have moved the data.
# We try the most-recent dataset first and fall back to the pilot folder.
CANDIDATE_DIRS = [
    Path("../data/dataset_28_04"),
    Path("../sample_data_labeled"),
]
DATA_DIR = next((p for p in CANDIDATE_DIRS if p.exists()), CANDIDATE_DIRS[0])
print(f"Using data directory: {DATA_DIR.resolve()}")

GEOJSON_FILES = sorted(DATA_DIR.glob("*.geojson"))
print(f"Found {len(GEOJSON_FILES)} GeoJSON files:")
for p in GEOJSON_FILES:
    print(f"  {p.name:32s}  ({p.stat().st_size/1024:7.1f} KB)")


In [ ]:
STAGE_PATTERN = re.compile(r"stage\s*(\d+)", re.IGNORECASE)


def extract_stage(props: dict) -> int | None:
    """Pull the stage number (0-4) out of an annotation's properties.

    Stage information lives in either ``properties.metadata.ANNOTATION_DESCRIPTION``
    (e.g. "Oosorption Stage 3") or the ``properties.name`` field.
    Returns ``None`` when no stage can be parsed.
    """
    candidates = []
    meta = props.get("metadata")
    if isinstance(meta, dict):
        desc = meta.get("ANNOTATION_DESCRIPTION")
        if desc:
            candidates.append(str(desc))
    name = props.get("name")
    if name:
        candidates.append(str(name))
    cls = props.get("classification")
    if isinstance(cls, dict) and cls.get("name"):
        candidates.append(str(cls["name"]))

    for text in candidates:
        m = STAGE_PATTERN.search(text)
        if m:
            return int(m.group(1))
    return None


def feature_to_polygon(feat: dict):
    """Convert a GeoJSON feature to a shapely polygon when possible."""
    geom = feat.get("geometry") or {}
    gtype = geom.get("type")
    if gtype in ("Polygon", "MultiPolygon"):
        try:
            return shape(geom)
        except Exception:
            return None
    if gtype == "LineString":
        coords = geom.get("coordinates", [])
        if len(coords) < 3:
            return None
        # close the ring if necessary
        if coords[0] != coords[-1]:
            coords = coords + [coords[0]]
        try:
            poly = Polygon(coords)
            if not poly.is_valid:
                poly = poly.buffer(0)
            return poly
        except Exception:
            return None
    return None


def parse_slice_id(stem: str) -> dict:
    """Parse file stem like ``LHP_SU_9_25-27`` into structured fields."""
    parts = stem.split("_")
    info = {"slice_id": stem}
    if len(parts) >= 4:
        info["location"] = parts[0]
        info["season"] = parts[1]
        info["colony"] = parts[2]
        info["sample"] = "_".join(parts[:3])
        info["slice_range"] = parts[3]
    return info


In [ ]:
records = []

for path in GEOJSON_FILES:
    with path.open("r", encoding="utf-8") as fp:
        gj = json.load(fp)

    slice_info = parse_slice_id(path.stem)

    for feat in gj.get("features", []):
        props = feat.get("properties", {}) or {}
        geom = feat.get("geometry") or {}

        stage = extract_stage(props)
        poly = feature_to_polygon(feat)

        record = {
            **slice_info,
            "feature_id": feat.get("id"),
            "geom_type": geom.get("type"),
            "stage": stage,
            "name": props.get("name"),
        }
        if poly is not None and not poly.is_empty:
            record["area"] = float(poly.area)
            record["perimeter"] = float(poly.length)
            minx, miny, maxx, maxy = poly.bounds
            record["bbox_width"] = maxx - minx
            record["bbox_height"] = maxy - miny
            record["centroid_x"] = poly.centroid.x
            record["centroid_y"] = poly.centroid.y
        else:
            record["area"] = np.nan
            record["perimeter"] = np.nan
            record["bbox_width"] = np.nan
            record["bbox_height"] = np.nan
            record["centroid_x"] = np.nan
            record["centroid_y"] = np.nan

        records.append(record)

df = pd.DataFrame.from_records(records)

# Tidy up the stage column
df["stage_label"] = df["stage"].apply(
    lambda s: f"Stage {int(s)}" if pd.notna(s) else "Unlabelled"
)

print(f"Loaded {len(df)} oocyte annotations from {df['slice_id'].nunique()} slice files.")
df.head()


## 1.  High-level summary

Quick sanity checks on the dataset: number of annotations per slice, overall
stage distribution, and the fraction of annotations that could not be
assigned a stage.


In [ ]:
summary = (
    df.groupby("slice_id")
      .agg(n_oocytes=("feature_id", "size"),
           n_unlabelled=("stage", lambda s: s.isna().sum()),
           geom_types=("geom_type", lambda s: dict(Counter(s))),
           stages=("stage_label", lambda s: dict(Counter(s))))
      .reset_index()
      .sort_values("n_oocytes", ascending=False)
)
summary


In [ ]:
print("Dataset totals")
print(f"  Total annotations:   {len(df)}")
print(f"  Slice files:         {df['slice_id'].nunique()}")
print(f"  Unique samples:      {df['sample'].nunique()}")
print(f"  Unlabelled features: {df['stage'].isna().sum()}")
print()
print("Feature geometry types:")
print(df['geom_type'].value_counts().to_string())
print()
print("Overall stage distribution:")
print(df['stage_label'].value_counts().sort_index().to_string())


## 2.  Oocytes per slice

How many oocytes were annotated in each slice file?


In [ ]:
counts_per_slice = (
    df.groupby("slice_id")
      .size()
      .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, max(3, 0.4 * len(counts_per_slice))))
counts_per_slice.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Number of annotated oocytes")
ax.set_ylabel("Slice file")
ax.set_title("Oocytes annotated per slice")
for i, v in enumerate(counts_per_slice.values):
    ax.text(v + 0.5, i, str(v), va="center")
plt.tight_layout()
plt.show()

counts_per_slice.describe()


## 3.  Overall stage distribution

Oosorption stage counts across the full dataset.  Stage 0 corresponds to
un-resorbed oocytes; higher stages indicate progressive resorption.


In [ ]:
stage_order = ["Stage 0", "Stage 1", "Stage 2", "Stage 3", "Stage 4", "Unlabelled"]
stage_counts = (
    df["stage_label"].value_counts()
      .reindex(stage_order)
      .dropna()
      .astype(int)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = sns.color_palette("viridis", n_colors=len(stage_counts))
stage_counts.plot.bar(ax=axes[0], color=colors, edgecolor="black")
axes[0].set_title("Oocyte counts by stage")
axes[0].set_ylabel("Number of oocytes")
axes[0].set_xlabel("Stage")
for i, v in enumerate(stage_counts.values):
    axes[0].text(i, v + 2, str(v), ha="center")

axes[1].pie(stage_counts.values, labels=stage_counts.index, colors=colors,
            autopct="%1.1f%%", startangle=90, pctdistance=0.75)
axes[1].set_title("Stage share")
axes[1].axis("equal")

plt.tight_layout()
plt.show()

stage_counts.to_frame("count").assign(
    percent=lambda d: (d["count"] / d["count"].sum() * 100).round(2)
)


## 4.  Stage composition per slice

A stacked view showing how each slice file breaks down by stage.  This is
useful for spotting samples that are biased toward early or late resorption.


In [ ]:
stage_by_slice = (
    df.groupby(["slice_id", "stage_label"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=[c for c in stage_order if c in df["stage_label"].unique()],
               fill_value=0)
)

# sort rows by total count ascending for nicer horizontal bars
stage_by_slice = stage_by_slice.loc[stage_by_slice.sum(axis=1).sort_values().index]

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, 0.45 * len(stage_by_slice))))

stage_by_slice.plot(kind="barh", stacked=True, ax=axes[0],
                    color=sns.color_palette("viridis", n_colors=stage_by_slice.shape[1]),
                    edgecolor="white")
axes[0].set_title("Absolute counts per slice")
axes[0].set_xlabel("Number of oocytes")
axes[0].legend(title="Stage", bbox_to_anchor=(1.02, 1), loc="upper left")

stage_frac = stage_by_slice.div(stage_by_slice.sum(axis=1), axis=0)
stage_frac.plot(kind="barh", stacked=True, ax=axes[1],
                color=sns.color_palette("viridis", n_colors=stage_frac.shape[1]),
                edgecolor="white")
axes[1].set_title("Relative composition per slice")
axes[1].set_xlabel("Fraction of oocytes")
axes[1].set_xlim(0, 1)
axes[1].legend(title="Stage", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

stage_by_slice


## 5.  Geometric properties of oocytes

The annotation coordinates are in full-resolution pixels of the source
NDPI slide (µm-per-pixel is recorded in the NDPI metadata if you need to
convert to physical units).  Here we look at annotated oocyte **area** and
**perimeter** to see whether the different stages differ systematically in
size.


In [ ]:
geom_df = df.dropna(subset=["area"]).copy()
geom_df["stage_label"] = pd.Categorical(
    geom_df["stage_label"],
    categories=[s for s in stage_order if s in geom_df["stage_label"].unique()],
    ordered=True,
)

stage_stats = (
    geom_df.groupby("stage_label", observed=True)
           .agg(n=("area", "size"),
                area_mean=("area", "mean"),
                area_median=("area", "median"),
                area_std=("area", "std"),
                perimeter_mean=("perimeter", "mean"),
                perimeter_median=("perimeter", "median"))
           .round(1)
)
stage_stats


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.boxplot(data=geom_df, x="stage_label", y="area", ax=axes[0],
            palette="viridis", showfliers=False)
sns.stripplot(data=geom_df, x="stage_label", y="area", ax=axes[0],
              color="black", size=2.5, alpha=0.5, jitter=0.25)
axes[0].set_yscale("log")
axes[0].set_title("Oocyte area by stage (log scale)")
axes[0].set_xlabel("Stage")
axes[0].set_ylabel("Area (px²)")

sns.boxplot(data=geom_df, x="stage_label", y="perimeter", ax=axes[1],
            palette="viridis", showfliers=False)
sns.stripplot(data=geom_df, x="stage_label", y="perimeter", ax=axes[1],
              color="black", size=2.5, alpha=0.5, jitter=0.25)
axes[1].set_yscale("log")
axes[1].set_title("Oocyte perimeter by stage (log scale)")
axes[1].set_xlabel("Stage")
axes[1].set_ylabel("Perimeter (px)")

plt.tight_layout()
plt.show()


In [ ]:
# Area histogram on a log scale, split per stage
fig, ax = plt.subplots(figsize=(9, 4.5))
for stage_label, sub in geom_df.groupby("stage_label", observed=True):
    if sub.empty:
        continue
    ax.hist(np.log10(sub["area"].clip(lower=1)),
            bins=30, alpha=0.55, label=stage_label, edgecolor="white")
ax.set_xlabel("log10(area in px²)")
ax.set_ylabel("Number of oocytes")
ax.set_title("Distribution of oocyte area per stage")
ax.legend(title="Stage")
plt.tight_layout()
plt.show()


## 6.  Aggregation by colony / location

Each slice file belongs to a coral specimen identified by the
`location_season_colony` prefix of its file name (e.g. `LHP_SU_9`,
`CHN_AU_8`).  Grouping annotations at this level lets us compare
between colonies and sampling seasons.


In [ ]:
sample_summary = (
    df.groupby(["location", "season", "sample"])
      .agg(n_slices=("slice_id", "nunique"),
           n_oocytes=("feature_id", "size"),
           stage0=("stage", lambda s: (s == 0).sum()),
           stage1=("stage", lambda s: (s == 1).sum()),
           stage2=("stage", lambda s: (s == 2).sum()),
           stage3=("stage", lambda s: (s == 3).sum()),
           stage4=("stage", lambda s: (s == 4).sum()),
           unlabelled=("stage", lambda s: s.isna().sum()))
      .reset_index()
      .sort_values("n_oocytes", ascending=False)
)
sample_summary


In [ ]:
sample_stage = sample_summary.set_index("sample")[
    ["stage0", "stage1", "stage2", "stage3", "stage4", "unlabelled"]
]
sample_stage.columns = ["Stage 0", "Stage 1", "Stage 2", "Stage 3", "Stage 4", "Unlabelled"]
sample_stage = sample_stage.loc[sample_stage.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(sample_stage))))
sample_stage.plot(kind="barh", stacked=True, ax=ax,
                  color=sns.color_palette("viridis", n_colors=sample_stage.shape[1]),
                  edgecolor="white")
ax.set_title("Oocyte stage composition per sample")
ax.set_xlabel("Number of oocytes")
ax.legend(title="Stage", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 7.  Spatial layout of annotations

For a quick look at where oocytes sit on each slide, plot their centroids
per slice, coloured by stage.  The axes are in raw slide pixels.


In [ ]:
plot_df = df.dropna(subset=["centroid_x", "centroid_y"])
slice_ids = sorted(plot_df["slice_id"].unique())
n = len(slice_ids)
cols = 2
rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(12, 3.5 * rows))
axes = np.atleast_1d(axes).ravel()

palette = dict(zip(stage_order, sns.color_palette("viridis", n_colors=len(stage_order))))

for ax, sid in zip(axes, slice_ids):
    sub = plot_df[plot_df["slice_id"] == sid]
    for label, chunk in sub.groupby("stage_label"):
        ax.scatter(chunk["centroid_x"], chunk["centroid_y"],
                   s=20, alpha=0.8,
                   c=[palette.get(label, "grey")], label=label,
                   edgecolor="black", linewidth=0.3)
    ax.invert_yaxis()  # image-style coordinates
    ax.set_title(f"{sid}  (n={len(sub)})", fontsize=10)
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
    ax.set_aspect("equal", adjustable="datalim")

# hide any trailing empty axes
for ax in axes[n:]:
    ax.set_visible(False)

# one shared legend
handles, labels = [], []
for label, colour in palette.items():
    handles.append(plt.Line2D([], [], marker="o", linestyle="",
                               markerfacecolor=colour, markeredgecolor="black",
                               markersize=7))
    labels.append(label)
fig.legend(handles, labels, title="Stage", loc="upper center",
           bbox_to_anchor=(0.5, 1.02), ncol=len(labels))
plt.tight_layout()
plt.show()


## 8.  YOLO bounding-box coverage analysis

To train an object detector such as YOLO on this dataset, the polygon
annotations have to be reduced to **axis-aligned bounding boxes**.
Because real oocytes are round / oval rather than rectangular, an
axis-aligned box is always larger than the underlying polygon and the
extra area is "background" (or, worse, parts of neighbouring oocytes)
that the detector will be trained to associate with the class.

For each annotation we compute:

*   `bbox_area`  – area of the tight axis-aligned bounding box.
*   `coverage`   – `polygon_area / bbox_area`, i.e. how much of the box
    is actually filled by the annotated oocyte.  A value of 1.0 means
    the polygon is itself a rectangle; a perfect circle inscribed in a
    square would give `π/4 ≈ 0.785`.
*   `aspect_ratio` – `bbox_width / bbox_height`, useful for choosing
    YOLO anchor priors.
*   `box_overlap` – per-slice fraction of bbox-area that overlaps with
    *another* oocyte's bbox.  High values warn that bbox-only training
    will entangle nearby instances.


In [ ]:
bbox_records = []

for path in GEOJSON_FILES:
    with path.open("r", encoding="utf-8") as fp:
        gj = json.load(fp)

    slice_info = parse_slice_id(path.stem)
    polys_in_slice = []  # collected for the overlap calculation below

    for feat in gj.get("features", []):
        props = feat.get("properties", {}) or {}
        poly = feature_to_polygon(feat)
        if poly is None or poly.is_empty:
            continue
        stage = extract_stage(props)
        minx, miny, maxx, maxy = poly.bounds
        bw = maxx - minx
        bh = maxy - miny
        bbox_area = bw * bh if bw > 0 and bh > 0 else np.nan
        coverage = poly.area / bbox_area if bbox_area else np.nan

        polys_in_slice.append((feat.get("id"), poly, (minx, miny, maxx, maxy)))

        bbox_records.append({
            **slice_info,
            "feature_id": feat.get("id"),
            "stage": stage,
            "stage_label": f"Stage {int(stage)}" if pd.notna(stage) else "Unlabelled",
            "polygon_area": poly.area,
            "bbox_x": minx,
            "bbox_y": miny,
            "bbox_w": bw,
            "bbox_h": bh,
            "bbox_area": bbox_area,
            "coverage": coverage,
            "aspect_ratio": bw / bh if bh > 0 else np.nan,
        })

bbox_df = pd.DataFrame.from_records(bbox_records)
print(f"Computed bounding boxes for {len(bbox_df)} annotations.")
bbox_df[[
    "slice_id", "stage_label", "polygon_area", "bbox_area",
    "coverage", "aspect_ratio",
]].head()


In [ ]:
# Per-stage coverage / aspect-ratio statistics
coverage_stats = (
    bbox_df.groupby("stage_label")
           .agg(n=("coverage", "size"),
                coverage_mean=("coverage", "mean"),
                coverage_median=("coverage", "median"),
                coverage_std=("coverage", "std"),
                coverage_min=("coverage", "min"),
                aspect_mean=("aspect_ratio", "mean"),
                aspect_median=("aspect_ratio", "median"))
           .round(3)
)
coverage_stats


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Coverage distribution overall + per stage
axes[0].hist(bbox_df["coverage"].dropna(), bins=30,
             color="steelblue", edgecolor="white")
axes[0].axvline(np.pi / 4, color="red", linestyle="--",
                label=r"perfect circle: $\pi/4 \approx 0.785$")
axes[0].axvline(bbox_df["coverage"].mean(), color="black", linestyle=":",
                label=f"mean = {bbox_df['coverage'].mean():.3f}")
axes[0].set_xlabel("polygon_area / bbox_area")
axes[0].set_ylabel("Number of oocytes")
axes[0].set_title("Bounding-box coverage – overall")
axes[0].legend()

stages_present = [s for s in stage_order if s in bbox_df["stage_label"].unique()]
sns.boxplot(data=bbox_df, x="stage_label", y="coverage", ax=axes[1],
            order=stages_present, palette="viridis", showfliers=False)
sns.stripplot(data=bbox_df, x="stage_label", y="coverage", ax=axes[1],
              order=stages_present, color="black", size=2.5,
              alpha=0.5, jitter=0.25)
axes[1].axhline(np.pi / 4, color="red", linestyle="--", alpha=0.7)
axes[1].set_title("Bounding-box coverage by stage")
axes[1].set_xlabel("Stage")
axes[1].set_ylabel("polygon_area / bbox_area")
axes[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.show()


In [ ]:
# Aspect ratio: how square / elongated are the boxes?
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(np.log2(bbox_df["aspect_ratio"].clip(lower=1e-3)),
        bins=30, color="darkorange", edgecolor="white")
ax.axvline(0, color="black", linestyle=":", label="square (1:1)")
ax.set_xlabel("log2(aspect ratio)  [width / height]")
ax.set_ylabel("Number of bounding boxes")
ax.set_title("Distribution of bbox aspect ratios")
ax.legend()
plt.tight_layout()
plt.show()

print("Aspect-ratio summary (width / height):")
print(bbox_df["aspect_ratio"].describe().round(3).to_string())


In [ ]:
# How much do the boxes overlap with each other within a slice?
# This matters because YOLO assigns each anchor to a single instance.
from shapely.geometry import box as shapely_box
from shapely.ops import unary_union

overlap_rows = []
for slice_id, sub in bbox_df.groupby("slice_id"):
    boxes = [shapely_box(r.bbox_x, r.bbox_y,
                          r.bbox_x + r.bbox_w, r.bbox_y + r.bbox_h)
             for r in sub.itertuples()]
    total_area = sum(b.area for b in boxes)
    if total_area == 0 or len(boxes) < 2:
        union_area = total_area
    else:
        union_area = unary_union(boxes).area
    # area covered by overlapping regions = sum - union (counting once)
    overlap_area = max(0.0, total_area - union_area)
    overlap_rows.append({
        "slice_id": slice_id,
        "n_boxes": len(boxes),
        "sum_box_area": total_area,
        "union_box_area": union_area,
        "overlap_area": overlap_area,
        "overlap_fraction": overlap_area / total_area if total_area else 0.0,
    })

overlap_df = (pd.DataFrame(overlap_rows)
                .sort_values("overlap_fraction", ascending=False))
overlap_df.round(4)


In [ ]:
# Box-size distribution – useful when picking YOLO input resolution / anchors
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(np.log10(bbox_df["bbox_w"].clip(lower=1)), bins=30,
             alpha=0.7, label="width", color="steelblue", edgecolor="white")
axes[0].hist(np.log10(bbox_df["bbox_h"].clip(lower=1)), bins=30,
             alpha=0.7, label="height", color="darkorange", edgecolor="white")
axes[0].set_xlabel("log10(pixels)  [slide-level 0]")
axes[0].set_ylabel("Number of boxes")
axes[0].set_title("Bounding-box width / height distribution")
axes[0].legend()

# Scatter of width vs height, coloured by stage
palette_b = dict(zip(stages_present,
                     sns.color_palette("viridis", n_colors=len(stages_present))))
for stage_label, sub in bbox_df.groupby("stage_label"):
    axes[1].scatter(sub["bbox_w"], sub["bbox_h"], s=15, alpha=0.75,
                    color=palette_b.get(stage_label, "grey"),
                    edgecolor="black", linewidth=0.2,
                    label=stage_label)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("bbox width (px)")
axes[1].set_ylabel("bbox height (px)")
axes[1].set_title("Box dimensions per stage")
axes[1].legend(title="Stage", fontsize=8)

plt.tight_layout()
plt.show()


### Implications for YOLO training

*   The dataset-wide mean bbox **coverage is ≈ 0.68** – noticeably
    below the `π/4 ≈ 0.785` value expected for a perfect ellipse,
    showing that the traced polygons are quite irregular.  In practice
    this means **roughly a third of every bounding box is non-oocyte
    tissue** that the detector will be asked to learn as foreground.
    An instance-segmentation head (YOLO-seg, Mask R-CNN) avoids this
    leakage entirely.
*   Coverage drops further for the later stages (Stage 2–4 average
    0.55 – 0.57), which have more elongated / lobed shapes.  This is
    exactly the regime where bounding boxes hurt the most.
*   Aspect ratios cluster around 1:1 (median ≈ 1.05) but with a long
    tail – Stage 4 boxes are the most elongated.  A single near-square
    anchor still works as a starting prior.
*   Per-slice bbox **overlap fractions reach ~20 %** on the densest
    slides (`LHP_W_10_28-30`); this is high enough that NMS thresholds
    and matcher tuning will matter.
*   Class imbalance remains the dominant problem: **87.6 % of all
    1 213 annotations are Stage 0**, and Stage 1 has only 6 instances.
    No detector can reliably learn that class from this corpus alone.

## 9.  Can YOLO be used here?  Alternatives.

**Short answer: yes – but a polygon-aware model is a better fit.**

YOLO (v8 / v11 in particular) is a fast, well-tooled multi-class
detector and would happily ingest the bounding boxes produced by the
companion script `scripts/geojson_to_yolo.py`.  Reasonable results are
achievable with transfer learning, heavy augmentation and class-balanced
sampling.

The caveats specific to this dataset are:

1. **Growing but still limited labelled corpus** – 1 213 instances across
   26 slides (up from 411 / 10 in the April pilot) is much better but still
   below what YOLO-from-scratch needs.  Pre-train on COCO / a related
   histology dataset and fine-tune.
2. **Severe class imbalance** – 87.6 % Stage 0; Stage 1 has only 6 examples.
   Combine class-balanced sampling with focal loss or re-frame as
   detect-then-classify.
3. **Whole-slide images are huge** – 165 k × 41 k pixels.  Tiles must
   be cut at training time and stitched at inference; the conversion
   script supports both whole-slide and per-tile YOLO labels.
4. **Round shapes** – axis-aligned boxes waste ~22 % of the cropped
   region.  Mask-aware models recover that signal.

### Alternative approaches, ordered by suitability

| Method | Output | Strength on this data | Notes |
|---|---|---|---|
| **YOLOv8-seg / YOLOv11-seg** | Polygon mask | Same trainer/ecosystem as YOLO, but uses the polygon | Drop-in replacement once polygons are exported. |
| **StarDist** | Star-convex polygon | Designed for round nuclei / oocytes; strong with few labels | Excellent for the Stage-0 class; needs separate stage classifier. |
| **CellPose 2 / CellPose-SAM** | Instance masks | Generalist cell segmentation, fine-tunes from a few labels | Very fast to bootstrap; pair with a small CNN classifier. |
| **HoVer-Net** | Instance + classification | Built for H&E nuclei; predicts type per instance | Heavier training but gives stage labels directly. |
| **Mask R-CNN (Detectron2)** | Instance mask + class | Mature, well-supported, polygon-friendly | More compute than YOLO but more accurate per-instance. |
| **U-Net + watershed / connected components** | Pixel mask, then post-processed | Simplest deep model, works on tiny datasets | Loses instance separation when oocytes touch. |
| **DETR / DINO** | Bounding box (or mask variant) | Set-prediction, no anchors | Data-hungry; usually overkill at this scale. |
| **Two-stage detect→classify** | Box + classifier crop | Decouples detection (easy, abundant labels) from staging (hard, few labels) | Robust to imbalance; recommended baseline. |
| **SAM2 prompt-based** | Mask | Few-shot, no training needed | Useful for active labelling but not for end-to-end deployment. |

**Recommended first experiment:** train **YOLOv8-seg** on
`stage` ⇒ {Stage 0, Stage 1-4 merged} with the polygons from
`data/dataset_28_04/`, then attach a lightweight EfficientNet-B0
classifier on the cropped instances to recover the stage label.  This
sidesteps both the bbox-coverage loss and the worst of the class
imbalance while staying inside a single, easy-to-deploy ecosystem.


## 10.  Key take-aways

The summaries above surface several clear patterns in the updated
`data/dataset_28_04` dataset (28 April 2026 — **1 213 oocytes across
26 slice files, 14 unique samples**):

*   Sample sizes are still uneven – several slice files contain more than
    100 oocytes while others contain fewer than ten.  Any downstream
    statistical model will need to account for this imbalance (e.g. through
    stratified sampling or class weights).
*   **Stage 0 still dominates** (87.6 % of all annotations), which is
    biologically expected; however, compared with the April pilot (91 %)
    the later stages are now better represented.  Stage 1 has grown from
    1 to 6 instances; Stage 3 (64) and Stage 4 (67) are now large enough
    to start modelling, but Stage 1 and Stage 2 (13 instances) remain too
    scarce for reliable classification.
*   **Zero unlabelled features** in the current dataset – every annotation
    carries a stage label, removing the need for defensive handling of
    missing classes.
*   Most annotations are still stored as traced `LineString` contours rather
    than filled `Polygon`s.  The notebook converts them to shapely polygons
    so that area / perimeter metrics can be computed consistently.
*   Area distributions shift with stage – later-stage oocytes are on average
    smaller (Stage 4 median bbox area < Stage 0), consistent with progressive
    resorption.  This size signal is a useful secondary feature for a stage
    classifier.

### Suggested next steps

1.  Convert the pixel-based area/perimeter values to physical units
    (µm² and µm) using the `mpp-x` / `mpp-y` fields from the NDPI
    metadata.  NDPI files can be downloaded with
    `scripts/download_ndpi.py` (CHN → PANGAEA 984641, LHP → PANGAEA 984640)
    and should be placed in `data/ndpi/`.
2.  Continue annotation toward the agreed ~100-cut / 1 000–2 000-oocyte
    target; prioritise active sampling of Stage 1 and Stage 2 oocytes.
3.  Run a first **YOLOv8-seg** baseline (detect-then-classify strategy) now
    that Stage 3 and Stage 4 counts are high enough to form a validation set.
4.  When exporting to YOLO format, prefer the *segmentation* variant
    (`scripts/geojson_to_yolo.py --task segment`) so the polygon shape is
    preserved.  The bbox coverage analysis above shows that ~32 % of every
    axis-aligned bounding box is background tissue.
